# 03 — Train Stage2

In [4]:
from pathlib import Path
import subprocess
import sys

import torch
from google.colab import drive


drive.mount("/content/drive")

project_dir = Path("/content/drive/MyDrive/Research/PUMA/PUMA Nuclei Pipeline")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", str(project_dir / "requirements.txt")],
    check=True,
)

sys.path.insert(0, str(project_dir))

from common.settings import paths as get_paths, plans

paths = get_paths()
assert torch.cuda.is_available(), "Select a Colab GPU runtime."

print("GPU:", torch.cuda.get_device_name(0))
print("Stage 1 cache:", paths["stage1_cache"])
print("Stage 2 cache:", paths["stage2_cache"])

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
GPU: NVIDIA L4
Stage 1 cache: /content/drive/MyDrive/Research/PUMA/PUMA Nuclei Pipeline/preprocess/stage1
Stage 2 cache: /content/drive/MyDrive/Research/PUMA/PUMA Nuclei Pipeline/preprocess/stage2


## Verify feature cache


In [5]:
import json


manifest = json.loads((paths["stage2_cache"] / "manifest.json").read_text())

assert manifest["nuclei"] == 97429
assert manifest["rois"] == 206

print("Stage 2 feature cache:", manifest)

Stage 2 feature cache: {'rois': 206, 'names': ['training_set_metastatic_roi_001', 'training_set_metastatic_roi_002', 'training_set_metastatic_roi_003', 'training_set_metastatic_roi_004', 'training_set_metastatic_roi_005', 'training_set_metastatic_roi_006', 'training_set_metastatic_roi_007', 'training_set_metastatic_roi_008', 'training_set_metastatic_roi_009', 'training_set_metastatic_roi_010', 'training_set_metastatic_roi_011', 'training_set_metastatic_roi_012', 'training_set_metastatic_roi_013', 'training_set_metastatic_roi_014', 'training_set_metastatic_roi_015', 'training_set_metastatic_roi_016', 'training_set_metastatic_roi_017', 'training_set_metastatic_roi_018', 'training_set_metastatic_roi_019', 'training_set_metastatic_roi_020', 'training_set_metastatic_roi_021', 'training_set_metastatic_roi_022', 'training_set_metastatic_roi_023', 'training_set_metastatic_roi_024', 'training_set_metastatic_roi_025', 'training_set_metastatic_roi_026', 'training_set_metastatic_roi_027', 'trainin

## Train/skip all 8 Stage 2 folds


In [6]:
import gc

from common.local_cache import local_copy_if_space
from stage2.train import train_one


training_cache = local_copy_if_space(
    paths["stage2_cache"],
    "/content/puma_stage2_cache",
    buffer_gib=3,
)

EPOCHS = 20

for plan in plans:
    for fold in range(4):
        print(f"STAGE 2 | {plan} | fold {fold}", flush=True)

        train_one(
            training_cache,
            paths["results"],
            plan,
            fold,
            epochs=EPOCHS,
            batch_size=0,
        )

        gc.collect()
        torch.cuda.empty_cache()

Using verified Colab-local cache: /content/puma_stage2_cache
STAGE 2 | A | fold 0
Stage 2 A/fold0 epoch 1/20 train_macro_f1=0.1521 val_macro_f1=0.1505 best=0.1505
Stage 2 A/fold0 epoch 2/20 train_macro_f1=0.3222 val_macro_f1=0.3232 best=0.3232
Stage 2 A/fold0 epoch 3/20 train_macro_f1=0.5463 val_macro_f1=0.5199 best=0.5199
Stage 2 A/fold0 epoch 4/20 train_macro_f1=0.5988 val_macro_f1=0.5565 best=0.5565
Stage 2 A/fold0 epoch 5/20 train_macro_f1=0.6282 val_macro_f1=0.5733 best=0.5733
Stage 2 A/fold0 epoch 6/20 train_macro_f1=0.6528 val_macro_f1=0.5923 best=0.5923
Stage 2 A/fold0 epoch 7/20 train_macro_f1=0.6719 val_macro_f1=0.6023 best=0.6023
Stage 2 A/fold0 epoch 8/20 train_macro_f1=0.6844 val_macro_f1=0.6096 best=0.6096
Stage 2 A/fold0 epoch 9/20 train_macro_f1=0.6984 val_macro_f1=0.6225 best=0.6225
Stage 2 A/fold0 epoch 10/20 train_macro_f1=0.7051 val_macro_f1=0.6275 best=0.6275
Stage 2 A/fold0 epoch 11/20 train_macro_f1=0.7171 val_macro_f1=0.6379 best=0.6379
Stage 2 A/fold0 epoch 12/